# DSCI 552 - Homework 3

**Name:** Yiming Xiong  
**USC ID:** 1057091668

## Import section

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### 1(a) Load the AReM Dataset

In [7]:
# Run this notebook from the repository root.
# The data are stored at HW3/data/AReM/<activity>/*.csv.
AREM_DIR = Path("../data/AReM")

if not AREM_DIR.is_dir():
    raise FileNotFoundError(
        f"Could not find the AReM data directory: {AREM_DIR}. "
        "Run the notebook from the repository root and keep the data in HW3/data/AReM/."
    )

records = []
for file_path in sorted(AREM_DIR.glob("*/*.csv")):
    records.append(
        {
            "activity": file_path.parent.name,
            "dataset_num": int(file_path.stem.replace("dataset", "")),
            "file_path": file_path,
        }
    )

manifest = pd.DataFrame(records)

activity_counts = (
    manifest.groupby("activity")
    .size()
    .rename("number_of_instances")
    .sort_index()
    .to_frame()
)

display(activity_counts)
print(f"Total instances: {len(manifest)}")

,number_of_instances
activity,
bending1,7
bending2,6
cycling,15
lying,15
sitting,15
standing,15
walking,15


Total instances: 88


### 1(b) Split the Training and Test Sets

In [8]:
bending_activities = {"bending1", "bending2"}

is_test = (
    (
        manifest["activity"].isin(bending_activities)
        & manifest["dataset_num"].isin([1, 2])
    )
    |
    (
        ~manifest["activity"].isin(bending_activities)
        & manifest["dataset_num"].isin([1, 2, 3])
    )
)

manifest["split"] = "train"
manifest.loc[is_test, "split"] = "test"

train_manifest = manifest.loc[manifest["split"] == "train"].reset_index(drop=True)
test_manifest = manifest.loc[manifest["split"] == "test"].reset_index(drop=True)

split_summary = (
    manifest.groupby(["split", "activity"])
    .size()
    .unstack(fill_value=0)
)

display(split_summary)
print(f"Training instances: {len(train_manifest)}")
print(f"Test instances: {len(test_manifest)}")

assert len(train_manifest) == 69, (
    f"Expected 69 training instances; found {len(train_manifest)}."
)
assert len(test_manifest) == 19, (
    f"Expected 19 test instances; found {len(test_manifest)}."
)

activity,bending1,bending2,cycling,lying,sitting,standing,walking
split,,,,,,,
test,2,2,3,3,3,3,3
train,5,4,12,12,12,12,12


Training instances: 69
Test instances: 19


### 1(c)(i) Common Time-Domain Features

Common time-domain features summarize the values and temporal behavior of a signal without transforming it into the frequency domain. Examples include:

- **Location:** mean, median, and quantiles.
- **Spread:** variance, standard deviation, range, and interquartile range.
- **Extremes:** minimum, maximum, and peak-to-peak amplitude.
- **Shape:** skewness and kurtosis.
- **Signal magnitude and change:** root mean square, signal energy, zero-crossing rate, mean absolute change, and linear trend.
- **Temporal dependence:** autocorrelation at one or more lags.

### 1(c)(ii) Extract the Required Features

The supplied data have three minor formatting inconsistencies: one file uses whitespace delimiters, and two files contain a missing line break that joins two observations into one physical line. The reader below handles these cases, then computes the seven required statistics for each of the six sensor series in every instance.

In [11]:
RAW_COLUMNS = [
    "time",
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23",
]

SENSOR_COLUMNS = RAW_COLUMNS[1:]
STATISTICS = ["min", "max", "mean", "median", "std", "q1", "q3"]


def read_activity_file(file_path):
    rows = []

    with open(file_path, "r", encoding="utf-8-sig") as file:
        for raw_line in file:
            line = raw_line.strip()

            if not line or line.startswith("#"):
                continue

            if "," in line:
                values = [value.strip() for value in line.split(",") if value.strip()]
            else:
                values = line.split()

            # A few rows contain two consecutive observations joined without a newline.
            if len(values) % len(RAW_COLUMNS) != 0:
                raise ValueError(
                    f"Unexpected number of values in {file_path}: "
                    f"found {len(values)} in line {line!r}"
                )

            for start in range(0, len(values), len(RAW_COLUMNS)):
                rows.append(values[start : start + len(RAW_COLUMNS)])

    frame = pd.DataFrame(rows, columns=RAW_COLUMNS)
    frame = frame.apply(pd.to_numeric, errors="raise")

    if frame.empty:
        raise ValueError(f"{file_path} contains no parsed observations.")

    return frame


def extract_instance_features(frame):
    feature_values = {}

    for column in SENSOR_COLUMNS:
        series = frame[column]

        feature_values[f"{column}_min"] = series.min()
        feature_values[f"{column}_max"] = series.max()
        feature_values[f"{column}_mean"] = series.mean()
        feature_values[f"{column}_median"] = series.median()
        feature_values[f"{column}_std"] = series.std(ddof=1)
        feature_values[f"{column}_q1"] = series.quantile(0.25)
        feature_values[f"{column}_q3"] = series.quantile(0.75)

    return feature_values


feature_records = []

for row in manifest.itertuples(index=False):
    activity_frame = read_activity_file(row.file_path)

    feature_record = {
        "instance": f"{row.activity}_dataset{row.dataset_num}",
        "activity": row.activity,
        "dataset_num": row.dataset_num,
    }
    feature_record.update(extract_instance_features(activity_frame))
    feature_records.append(feature_record)

feature_table = pd.DataFrame(feature_records)
FEATURE_COLUMNS = [
    f"{sensor}_{statistic}"
    for sensor in SENSOR_COLUMNS
    for statistic in STATISTICS
]

print(f"Feature table shape: {feature_table.shape}")
print(f"Number of extracted features: {len(FEATURE_COLUMNS)}")
display(feature_table)

Feature table shape: (88, 45)
Number of extracted features: 42


,instance,activity,dataset_num,avg_rss12_min,avg_rss12_max,avg_rss12_mean,avg_rss12_median,avg_rss12_std,avg_rss12_q1,avg_rss12_q3,...,avg_rss23_std,avg_rss23_q1,avg_rss23_q3,var_rss23_min,var_rss23_max,var_rss23_mean,var_rss23_median,var_rss23_std,var_rss23_q1,var_rss23_q3
0,bending1_dataset1,bending1,1,37.25,45.00,40.624792,40.500,1.476967,39.25,42.0000,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.430,0.582915,0.00,1.3000
1,bending1_dataset2,bending1,2,38.00,45.67,42.812812,42.500,1.435550,42.00,43.6700,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.430,0.601010,0.00,1.3000
2,bending1_dataset3,bending1,3,35.00,47.40,43.954500,44.330,1.558835,43.00,45.0000,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.430,0.513506,0.00,0.9400
3,bending1_dataset4,bending1,4,33.00,47.75,42.179812,43.500,3.670666,39.15,45.0000,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.500,0.524317,0.00,1.0000
4,bending1_dataset5,bending1,5,33.00,45.75,41.678063,41.750,2.243490,41.33,42.7500,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.430,0.389164,0.00,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,walking_dataset5,walking,5,20.75,46.25,34.763333,35.290,4.742208,31.67,38.2500,...,3.174681,14.2500,18.33,0.0,9.39,3.288271,3.270,1.647528,2.05,4.3050
84,walking_dataset6,walking,6,21.50,51.00,34.935812,35.500,4.645944,32.00,38.0625,...,3.192058,14.2375,18.25,0.0,10.21,3.280021,3.015,1.700918,2.12,4.5000
85,walking_dataset7,walking,7,18.33,47.67,34.333042,34.750,4.948770,31.25,38.0000,...,3.000493,13.7500,18.00,0.0,8.01,3.261583,2.980,1.617290,2.05,4.3200
86,walking_dataset8,walking,8,18.33,45.75,34.599875,35.125,4.731790,31.50,38.0000,...,2.905688,14.0000,18.25,0.0,8.86,3.289542,3.015,1.680170,2.12,4.2600


### 1(c)(iii) Standard Deviations and 90% Bootstrap Confidence Intervals

For each extracted feature, the point estimate is the sample standard deviation across the 88 instances. A nonparametric bootstrap resamples the instances with replacement 10,000 times. The 5th and 95th percentiles of the bootstrap standard deviations form a two-sided 90% percentile confidence interval.

In [12]:
N_BOOTSTRAP = 8000
RANDOM_SEED = 514

rng = np.random.default_rng(RANDOM_SEED)
bootstrap_rows = []

for feature_name in FEATURE_COLUMNS:
    values = feature_table[feature_name].to_numpy(dtype=float)
    point_estimate = values.std(ddof=1)

    resampled_values = rng.choice(
        values,
        size=(N_BOOTSTRAP, len(values)),
        replace=True,
    )
    bootstrap_sds = resampled_values.std(axis=1, ddof=1)
    lower_bound, upper_bound = np.quantile(bootstrap_sds, [0.05, 0.95])

    bootstrap_rows.append(
        {
            "feature": feature_name,
            "standard_deviation": point_estimate,
            "90%_CI_lower": lower_bound,
            "90%_CI_upper": upper_bound,
        }
    )

bootstrap_summary = pd.DataFrame(bootstrap_rows).set_index("feature")
display(bootstrap_summary)

,standard_deviation,90%_CI_lower,90%_CI_upper
feature,,,
avg_rss12_min,9.569975,8.266347,10.791800
avg_rss12_max,4.394362,3.317368,5.285684
avg_rss12_mean,5.335718,4.702846,5.899710
avg_rss12_median,5.440054,4.800715,6.006702
avg_rss12_std,1.772153,1.573080,1.947814
avg_rss12_q1,6.153590,5.575168,6.661934
avg_rss12_q3,5.138925,4.339264,5.840376
var_rss12_min,0.000000,0.000000,0.000000
var_rss12_max,5.062729,4.608999,5.403213


### 1(c)(iv) Select Three Important Feature Types

- **Mean (Average)**: The typical sensor reading, representing the overall level of the signal.
- **Standard deviation (Deviation)**: The degree of fluctuation within an individual instance.
- **Maximum (Peak)**: The peak value reached by the signal, reflecting strong instantaneous activity.

These three features describe the signal from three perspectives: overall level, fluctuation, and peak. They are initially selected based on their meanings; after being calculated separately on six sensor sequences, 18 feature columns will be obtained.

## 2. ISLR 3.7.4

This exercise compares linear and cubic regression using training and test residual sum of squares (RSS). The cubic model includes the linear model as a special case by setting the quadratic and cubic coefficients to zero.

### (a)

The cubic model’s training RSS will be less than or equal to the linear model’s training RSS. The cubic model can act just like a linear model by setting the coefficients of (X^2) and (X^3) to zero. So it has all the same fitting options as the linear model, plus some extra ones. Least squares picks the fit with the lowest RSS, which means the cubic model cannot end up with a higher training RSS. It may get a lower RSS by also fitting some noise in the training data, or it may tie with the linear model.

### (b)

We would generally expect the linear model to have a lower test RSS. Since the true relationship is linear, the extra cubic terms are not needed. They may fit random noise in the training data, a problem called overfitting, and that can hurt performance on new data. The test RSS from one particular sample can still vary, but the linear model is the better expectation here.

### (c)

The cubic model’s training RSS will still be less than or equal to the linear model’s training RSS. This is true even if we do not know what the true relationship looks like: the cubic model still includes the linear model as a special case, so adding fitting options cannot increase the lowest training RSS.

### (d)

There is not enough information to say which model will have a lower test RSS. If the true nonlinear pattern is close to a cubic curve, the cubic model may predict better. If it is not, the extra terms may not help and could lead to overfitting. The result depends on the shape of the true relationship and how well each model fits new data.